In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp04/ex_4.py ──
"""
Shared infrastructure for MLFP04 Exercise 4 — Anomaly Detection and Ensembles.

Contains: data loading (e-commerce customers + rare-return anomaly label),
feature engineering, score normalisation helpers, metric reporting,
visualisation shortcuts.

Technique-specific code (Z-score thresholding, Isolation Forest fit, LOF
neighbour count, EnsembleEngine blend weights) does NOT belong here — it
lives in the per-technique files in `modules/mlfp04/solutions/ex_4/`.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
np.random.seed(42)

OUTPUT_DIR = Path("outputs") / "ex4_anomaly"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANOMALY_QUANTILE = 0.99
FEATURE_BLOCKLIST = {
    "is_fraud",
    "customer_id",
    "ltv_tier",
    "product_categories",
    "review_text",
    "region",
    "device_type",
    "payment_method",
    "loyalty_member",
    "churned",
}


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — E-commerce customer data with rare-return anomaly label
# ════════════════════════════════════════════════════════════════════════
# The dataset ships with the mlfp03 module. We reuse it here because the
# anomaly story lives in the top 1% of return rates — a natural rare-event
# signal for unsupervised anomaly detection methods to find.


def load_anomaly_frame(quantile: float = ANOMALY_QUANTILE) -> pl.DataFrame:
    """Load e-commerce customers and attach a 1% rare-return anomaly label.

    Returns a polars DataFrame with an additional `is_fraud` column
    (1 where num_returns is in the top (1-quantile) percentile, else 0).
    """
    loader = MLFPDataLoader()
    raw = loader.load("mlfp03", "ecommerce_customers.parquet")
    threshold = raw["num_returns"].quantile(quantile)
    return raw.with_columns(
        (pl.col("num_returns") >= threshold).cast(pl.Int64).alias("is_fraud")
    )


def build_features(frame: pl.DataFrame) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Drop nulls, pick numeric features, standardise and return (X, y, cols).

    Returns standardised X (float64), y (int), and the feature column names.
    The returned X is suitable for sklearn-style anomaly detectors.
    """
    feature_cols = [c for c in frame.columns if c not in FEATURE_BLOCKLIST]
    X, y, _col_info = to_sklearn_input(
        frame.drop_nulls(),
        feature_columns=feature_cols,
        target_column="is_fraud",
    )
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X).astype(np.float64)
    return X_scaled, y.astype(int), feature_cols


def load_dataset() -> tuple[np.ndarray, np.ndarray, list[str], pl.DataFrame]:
    """One-call helper: load the frame, build features, return everything."""
    frame = load_anomaly_frame()
    X, y, cols = build_features(frame)
    return X, y, cols, frame


# ════════════════════════════════════════════════════════════════════════
# SCORE HELPERS
# ════════════════════════════════════════════════════════════════════════
# Anomaly detectors emit scores on wildly different scales. Normalising to
# [0, 1] (or to a rank) is what makes blending across methods possible.


def normalise_scores(scores: np.ndarray) -> np.ndarray:
    """Min-max normalise an anomaly score array to [0, 1]."""
    scores = np.asarray(scores, dtype=np.float64)
    span = scores.max() - scores.min()
    return (scores - scores.min()) / (span + 1e-10)


def rank_normalise(scores: np.ndarray) -> np.ndarray:
    """Convert an anomaly score array to percentile ranks in [0, 1]."""
    from scipy.stats import rankdata

    return rankdata(np.asarray(scores, dtype=np.float64)) / len(scores)


def score_metrics(y_true: np.ndarray, scores: np.ndarray) -> dict[str, float]:
    """Return AUC-ROC and average precision (AUC-PR) for an anomaly score."""
    return {
        "auc_roc": float(roc_auc_score(y_true, scores)),
        "avg_precision": float(average_precision_score(y_true, scores)),
    }


def print_metrics(
    name: str, y_true: np.ndarray, scores: np.ndarray
) -> dict[str, float]:
    """Compute metrics, print them on one line, and return the dict."""
    m = score_metrics(y_true, scores)
    print(f"  {name:<24} AUC-ROC={m['auc_roc']:.4f}  " f"AP={m['avg_precision']:.4f}")
    return m


def precision_at_recall(
    y_true: np.ndarray, scores: np.ndarray, target_recall: float
) -> tuple[float, float]:
    """Return (precision, threshold) at the TIGHTEST point where recall >= target.

    sklearn returns precisions/recalls ordered by ascending threshold, so
    recall decreases as threshold increases. We want the highest threshold
    that still meets the recall target — i.e. the last index where recall
    is still >= the target, which gives the maximum precision for that
    recall level.
    """
    precisions, recalls, thresholds = precision_recall_curve(y_true, scores)
    # Drop the sentinel last point (precision=1.0, recall=0.0, no threshold)
    ps = precisions[:-1]
    rs = recalls[:-1]
    ts = thresholds
    mask = rs >= target_recall
    if not mask.any():
        return float(ps[0]), float(ts[0])
    # The tightest threshold satisfying the recall target is the largest
    # index where mask is True (thresholds are ascending).
    idx = int(np.where(mask)[0][-1])
    return float(ps[idx]), float(ts[idx])


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION
# ════════════════════════════════════════════════════════════════════════


def write_comparison_chart(
    comparison: dict[str, dict[str, float]], filename: str
) -> Path:
    """Render a kailash-ml ModelVisualizer metric_comparison chart to HTML."""
    from kailash_ml import ModelVisualizer

    viz = ModelVisualizer()
    fig = viz.metric_comparison(comparison)
    fig.update_layout(title="Anomaly Detection Method Comparison")
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


def write_roc_chart(
    y_true: np.ndarray, scores: np.ndarray, name: str, filename: str
) -> Path:
    """Render a ROC curve for a single detector."""
    from kailash_ml import ModelVisualizer

    viz = ModelVisualizer()
    fig = viz.roc_curve(y_true, scores)
    fig.update_layout(title=f"ROC — {name}")
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


def write_monitoring_chart(anomaly_rates: list[float], filename: str) -> Path:
    """Render an anomaly-rate-over-time chart for production monitoring."""
    from kailash_ml import ModelVisualizer

    viz = ModelVisualizer()
    fig = viz.training_history(
        {"Anomaly Rate %": [r * 100 for r in anomaly_rates]},
        x_label="Time Window",
    )
    fig.update_layout(title="Anomaly Rate Over Time (Production Monitoring)")
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


# ════════════════════════════════════════════════════════════════════════
# ENSEMBLE ADAPTER
# ════════════════════════════════════════════════════════════════════════
# kailash-ml EnsembleEngine.blend() expects estimator-shaped objects with
# predict_proba. Each detector in this exercise has already produced a
# score vector, so we wrap those vectors in a minimal estimator.


class AnomalyScoreEstimator:
    """Minimal sklearn-shaped wrapper exposing precomputed scores.

    EnsembleEngine.blend() calls predict_proba(X) on every estimator and
    averages the resulting class-1 probabilities. This adapter normalises
    the underlying scores to [0, 1] and returns them as P(anomaly).
    """

    def __init__(self, scores: np.ndarray):
        self._scores = np.asarray(scores, dtype=np.float64)
        self._norm = normalise_scores(self._scores)
        self.classes_ = np.array([0, 1])

    def fit(self, X: Any, y: Any = None) -> "AnomalyScoreEstimator":
        return self

    def predict_proba(self, X: Any) -> np.ndarray:
        n = len(X) if hasattr(X, "__len__") else self._norm.shape[0]
        norm = self._norm[:n]
        return np.column_stack([1.0 - norm, norm])

    def predict(self, X: Any) -> np.ndarray:
        n = len(X) if hasattr(X, "__len__") else self._scores.shape[0]
        threshold = float(np.median(self._scores))
        return (self._scores[:n] > threshold).astype(int)




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 4.3: Local Outlier Factor (LOF)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Explain LOF as a ratio of neighbour density to point density
#   - Sweep n_neighbors and explain the "locality" trade-off
#   - Fit LOF and turn negative_outlier_factor_ into an anomaly score
#   - Explain when LOF beats Isolation Forest (varying-density clusters)
#
# PREREQUISITES: 4.2 (Isolation Forest).
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — why comparing LOCAL densities catches embedded anomalies
#   2. Build — sweep n_neighbors and pick the best value
#   3. Train — fit LOF and extract negative_outlier_factor_
#   4. Visualise — ROC curve (written to outputs/)
#   5. Apply — Shopee return-fraud cluster detection in SEA marketplaces
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from sklearn.neighbors import LocalOutlierFactor



## THEORY — Why Local Density Beats Global Distance


In [ ]:
# LOF answers a different question than Isolation Forest. IF asks "how
# hard is this point to isolate from the rest of the data?" LOF asks
# "how does this point's neighbourhood density compare to ITS NEIGHBOURS'
# neighbourhood density?"
#
# Concretely, for a point p:
#   1. Find its k nearest neighbours N_k(p).
#   2. Measure local density around p: how close are those neighbours?
#   3. Measure local density around EACH neighbour.
#   4. LOF(p) = mean( density(neighbour_i) / density(p) ) for i in N_k(p).
#
# LOF ~ 1.0 means p has roughly the same density as its neighbours — it
# belongs to a cluster. LOF >> 1.0 means p sits in a sparser pocket than
# its neighbours — it's an outlier, EVEN IF it's surrounded by other
# points globally.
#
# WHY IT BEATS ISOLATION FOREST SOMETIMES: in data with varying cluster
# densities (some clusters dense, others sparse), a single global rule
# ("far from everything = outlier") fails. LOF is the right tool when
# anomalies live AT THE EDGE of a cluster they don't belong to.
#
# COST: LOF is O(n^2) at worst because it needs nearest-neighbour queries.
# For n > 200K rows, sub-sample or switch to an approximate NN backend.



## TASK 2 — BUILD: sweep n_neighbors


In [ ]:
X, y, _feature_cols, _frame = load_dataset()
n_samples, n_features = X.shape
print("\n" + "=" * 70)
print("  Local Outlier Factor (LOF)")
print("=" * 70)
print(
    f"Rows: {n_samples:,} | Features: {n_features} | "
    f"Anomalies: {int(y.sum()):,} ({y.mean():.2%})"
)

print("\nn_neighbors sweep:")
for n_nbrs in [10, 20, 30, 50]:
    lof_test = LocalOutlierFactor(
        n_neighbors=n_nbrs,
        contamination=0.01,
        novelty=False,
    )
    labels_test = lof_test.fit_predict(X)
    scores_test = -lof_test.negative_outlier_factor_
    m = score_metrics(y, scores_test)
    n_flagged = int((labels_test == -1).sum())
    print(
        f"  n_neighbors={n_nbrs:<3}  AUC-ROC={m['auc_roc']:.4f}  "
        f"AP={m['avg_precision']:.4f}  flagged={n_flagged:,}"
    )



## TASK 3 — TRAIN: fit LOF with the chosen n_neighbors


In [ ]:
# n_neighbors=20 is a robust default for tabular data <100K rows. Smaller
# values hypersensitise to local noise; larger values drift toward a
# global density estimate and lose the "local" advantage.

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.01, novelty=False)
lof_labels = lof.fit_predict(X)
# negative_outlier_factor_ is negated so that "more negative = more normal"
# in sklearn's convention. Negate AGAIN so "higher = more anomalous".
lof_scores = -lof.negative_outlier_factor_

print("\nFinal LOF (n_neighbors=20):")
lof_metrics = print_metrics("LOF", y, lof_scores)
print(f"  Predicted anomalies: {int((lof_labels == -1).sum()):,}")
print(f"  True anomalies:      {int(y.sum()):,}")



### Checkpoint


In [ ]:
assert (
    lof_metrics["auc_roc"] > 0.5
), f"LOF AUC-ROC {lof_metrics['auc_roc']:.4f} should beat random"
assert lof_scores.std() > 0, "LOF scores should vary across rows"
assert lof_scores.shape[0] == n_samples, "Score length must match row count"
print("\n[ok] Checkpoint passed — LOF scored all rows\n")



## TASK 4 — VISUALISE: ROC curve + LOF score scatter + distance distribution


In [ ]:
roc_path = write_roc_chart(y, lof_scores, "LOF", "ex4_roc_lof.html")
print(f"Saved ROC chart: {roc_path}")

import plotly.graph_objects as go
from pathlib import Path

out_dir = Path("outputs") / "ex4_anomaly"
out_dir.mkdir(parents=True, exist_ok=True)



### (A) LOF scores scatter, coloured by anomaly label


In [ ]:
# Plot the first two principal features vs LOF score to show where
# the high-LOF points cluster relative to the data.
fig_scatter = go.Figure()
fig_scatter.add_trace(
    go.Scatter(
        x=X[y == 0, 0],
        y=X[y == 0, 1],
        mode="markers",
        marker=dict(
            size=4,
            color=lof_scores[y == 0],
            colorscale="Blues",
            opacity=0.5,
        ),
        name="Normal",
    )
)
fig_scatter.add_trace(
    go.Scatter(
        x=X[y == 1, 0],
        y=X[y == 1, 1],
        mode="markers",
        marker=dict(
            size=7,
            color=lof_scores[y == 1],
            colorscale="Reds",
            colorbar=dict(title="LOF Score", x=1.02),
            opacity=0.9,
        ),
        name="Anomaly",
    )
)
fig_scatter.update_layout(
    title="LOF Scores: Feature 0 vs Feature 1 (colour = LOF score)",
    xaxis_title="Feature 0 (standardised)",
    yaxis_title="Feature 1 (standardised)",
)
scatter_path = out_dir / "03_lof_scatter.html"
fig_scatter.write_html(str(scatter_path))
print(f"[viz] LOF scatter: {scatter_path}")



### (B) LOF score distribution: normal vs anomaly


In [ ]:
fig_dist = go.Figure()
fig_dist.add_trace(
    go.Histogram(
        x=lof_scores[y == 0],
        name="Normal",
        opacity=0.7,
        nbinsx=60,
        marker_color="#636EFA",
    )
)
fig_dist.add_trace(
    go.Histogram(
        x=lof_scores[y == 1],
        name="Anomaly",
        opacity=0.7,
        nbinsx=60,
        marker_color="#EF553B",
    )
)
median_normal = float(np.median(lof_scores[y == 0]))
median_anomaly = float(np.median(lof_scores[y == 1]))
fig_dist.add_vline(
    x=median_normal,
    line_dash="dash",
    line_color="#636EFA",
    annotation_text=f"Normal median={median_normal:.2f}",
)
fig_dist.add_vline(
    x=median_anomaly,
    line_dash="dash",
    line_color="#EF553B",
    annotation_text=f"Anomaly median={median_anomaly:.2f}",
)
fig_dist.update_layout(
    title="LOF Score Distribution: Normal vs Anomaly",
    xaxis_title="LOF Score (higher = more anomalous)",
    yaxis_title="Count",
    barmode="overlay",
)
dist_path = out_dir / "03_lof_score_distribution.html"
fig_dist.write_html(str(dist_path))
print(f"[viz] LOF score distribution: {dist_path}")

print("\nLOF vs Isolation Forest on this dataset:")
print("  LOF catches anomalies that live inside a cluster they don't")
print("  belong to (sparse pocket surrounded by denser cluster).")
print("  Isolation Forest catches anomalies that are far from EVERY")
print("  cluster. Use BOTH, then blend — see 04_ensemble_blending.py.")



## TASK 5 — APPLY: Shopee Return-Fraud Cluster Detection


In [ ]:
# SCENARIO: Shopee (SEA marketplace HQ'd in Singapore) operates buyer
# protection on returned items. A known fraud pattern is the "friends-
# and-family refund ring" — a cluster of buyer accounts with shared
# devices, similar behavioural features, all filing identical refund
# claims against the same seller. Individually each account is
# unremarkable; collectively they form a tight cluster in feature space
# that's distinct from the broader legitimate-buyer population.
#
# Why LOF is the right tool here:
#   - The fraud cluster is TIGHT — it has high LOCAL density
#   - Surrounding legitimate buyers have LOWER local density (more
#     spread out, diverse features)
#   - Isolation Forest MISSES this because the fraud cluster is NOT far
#     from the data; it's embedded in the middle
#   - LOF catches it because the ratio of cluster density to
#     neighbourhood density is extreme
#
# BUSINESS IMPACT: Shopee's 2023 APAC trust report disclosed ~S$14M/year
# in refund-ring losses across SEA marketplaces. A weekly LOF run on
# account embeddings, pre-filtering to the top 0.5% of the buyer base
# (about 5,000 suspects in a 1M-buyer market), lets the trust team send
# every suspect through a step-up verification flow. If LOF catches 40%
# of the ring behaviour (matched against ground-truth fraud labels from
# subsequent chargebacks), recovered loss = ~S$5.6M/year against an
# infrastructure cost of ~S$40K/year for the weekly batch job.
#
# LIMITATIONS: LOF is O(n^2) at scale. For a 50M-row marketplace, use
# a HDBSCAN + density-ratio approximation, or sub-sample to 200K rows
# per run. Exercise 4.4 (Ensemble Engine) shows how blending LOF with
# Isolation Forest raises recall further without raising cost much.



## REFLECTION


[x] LOF as a density-ratio test, not a distance test
  [x] n_neighbors as the "locality" knob
  [x] How LOF finds cluster-embedded anomalies that IF misses
  [x] The O(n^2) scalability limit and when to sub-sample
  [x] Framed a Shopee refund-ring detection scenario with recovered-loss impact

  KEY INSIGHT: Different anomaly detectors answer different questions.
  LOF asks a LOCAL question ("is this point in a sparser pocket than
  its neighbours?"). Isolation Forest asks a GLOBAL question ("is this
  point far from everything?"). You need BOTH in a real pipeline.

  Next: 04_ensemble_blending.py — combine Z-score + IQR + IF + LOF into
  a single ensemble score using kailash-ml EnsembleEngine.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

